# 06. 머신러닝 - 다중분류

> 이진분류(06번 파일)와 모델 코드 구조는 비슷하지만 평가함수 옵션·해석이 달라 별도 파일로 분리했습니다. Wine 데이터셋(포도주 품종 3종 분류)으로 실습합니다.

### 이 노트북에서 배우는 것
- 이진분류와 다중분류가 무엇이 다른지 다시 한번 짚고 시작하기
- 동일 모델군을 다중분류 데이터에 적용하기
- average='macro'/'weighted' 등 다중분류 전용 평가 옵션
- 다중클래스 confusion_matrix 해석

### 사용 데이터
- `data/wine.csv` (sklearn 내장 Wine 데이터셋, target 0/1/2 3개 클래스)

## 목차
1. 다중분류란? 이진분류와 무엇이 다른가
2. 다중분류 모델·모듈·평가함수 총정리표
3. 다중분류 모델 실습
4. 다중분류 평가지표
5. 다중클래스 confusion_matrix 시각화
6. 앙상블 개념 복습 + 다중분류 실전 비교 (배깅 vs 부스팅)
7. Feature Importance 해석
8. 여러 모델 앙상블 (VotingClassifier)
9. 여러 모델 성능 비교 시각화
10. 소프트맥스 직관 & 다중클래스 레이블 인코딩
11. 종합 실전 문제

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

wine = pd.read_csv('../data/wine.csv')
print(wine['target'].value_counts())
wine.head()

In [ ]:
X = wine.drop(columns=['target'])
y = wine['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train.shape, X_test.shape

## 1. 다중분류란? 이진분류와 무엇이 다른가

### 📖 이론 설명
다중분류는 클래스가 3개 이상인 경우입니다(여기서는 와인 품종 0, 1, 2). 06_머신러닝_분류_이진.ipynb(이진분류)과 비교하면 두 가지가 명확히 달라집니다.

1. **precision/recall/f1을 '하나의 숫자'로 요약하는 방법을 지정해야 합니다.** 클래스별로 각각 precision/recall이 나오기 때문에, 이를 합치는 방식을 `average` 옵션으로 정해야 합니다.
   - `'macro'`: 클래스별 점수를 **단순 평균** (클래스 크기 무시, 소수 클래스도 동등하게 취급)
   - `'weighted'`: 클래스별 데이터 개수로 **가중 평균** (불균형할 때 다수 클래스 영향이 큼)
   - `'micro'`: 전체 TP/FP/FN을 합산해서 계산 (전체적으로는 accuracy와 비슷해짐)
2. **confusion matrix가 2x2가 아니라 NxN**이 됩니다(N=클래스 수). 대각선이 아닌 칸은 모두 '어떤 클래스를 어떤 클래스로 잘못 예측했는지'를 보여줍니다.

모델을 만들고 학습시키는 코드(`fit`, `predict`) 자체는 06번과 동일합니다 - **바뀌는 것은 평가 단계**입니다.

## 2. 다중분류 모델·모듈·평가함수 총정리표

### 🧩 핵심 개념 정리
이진분류에서 썼던 모델들을 그대로 다중분류에도 쓸 수 있습니다(LogisticRegression, DecisionTree, RandomForest, XGBoost, LGBM, KNN, SVC 등). 달라지는 것은 평가함수의 옵션입니다.

| 평가함수 | 다중분류에서 반드시 필요한 옵션 |
|---|---|
| precision_score / recall_score / f1_score | `average='macro'` 또는 `'weighted'` |
| classification_report | 옵션 없이도 클래스별 결과를 모두 보여줌 |
| roc_auc_score | `multi_class='ovr'`, `average=...` 필요 |

## 3. 다중분류 모델 실습

### 📖 이론 설명
이진분류와 완전히 동일한 템플릿으로 여러 모델을 학습시킵니다. 클래스가 3개라는 것을 모델이 자동으로 인식해서 내부적으로 처리해줍니다.

### 💻 예제 코드

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000),
    'DecisionTree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss'),
}

preds = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds[name] = model.predict(X_test_scaled)
print('학습 완료:', list(models.keys()))

### ✍️ TODO: 실전 문제

**문제 1.** `KNeighborsClassifier(n_neighbors=5)`를 학습시키고 예측값 `knn_pred`를 만드세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.neighbors import KNeighborsClassifier
knn = KNeighborsClassifier(n_neighbors=5)  # n_neighbors=5: 가장 가까운 5개의 이웃을 참고해 다수결로 클래스를 정함
knn.fit(X_train_scaled, y_train)  # 거리 기반 모델이라 스케일링된 데이터를 사용하는 것이 특히 중요함
knn_pred = knn.predict(X_test_scaled)
```

</details>

## 4. 다중분류 평가지표

### 📖 이론 설명
이진분류와 똑같은 함수를 쓰지만 `average` 옵션을 꼭 지정해야 에러 없이 계산됩니다. `classification_report`는 옵션 없이도 클래스별 점수를 모두 보여주므로 다중분류에서 가장 먼저 찍어보기 좋은 함수입니다.

### 🧩 핵심 개념 정리
| 옵션값 | 의미 |
|---|---|
| average='macro' | 클래스별 점수의 단순 평균 |
| average='weighted' | 클래스 비율로 가중 평균 |

### 💻 예제 코드

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

rf_pred = preds['RandomForest']
print('accuracy:', accuracy_score(y_test, rf_pred))
print('f1 (macro):', f1_score(y_test, rf_pred, average='macro'))
print('f1 (weighted):', f1_score(y_test, rf_pred, average='weighted'))
print(classification_report(y_test, rf_pred))

### ✍️ TODO: 실전 문제

**문제 1.** `XGBoost` 예측 결과의 f1-score를 `average='macro'`와 `average='weighted'` 두 가지로 각각 구해 비교하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
xgb_pred = preds['XGBoost']
print('macro:', f1_score(y_test, xgb_pred, average='macro'))  # macro: 클래스 크기와 상관없이 단순 평균(소수 클래스도 동등하게 취급)
print('weighted:', f1_score(y_test, xgb_pred, average='weighted'))  # weighted: 클래스별 데이터 개수로 가중 평균
```

</details>

## 5. 다중클래스 confusion_matrix 시각화

### 📖 이론 설명
클래스가 3개면 혼동행렬은 3x3이 됩니다. 대각선(왼쪽 위→오른쪽 아래)이 '맞게 예측한 개수'이고, 대각선 밖의 값은 '어떤 클래스를 어떤 클래스로 착각했는지'를 보여줍니다. 예를 들어 (실제=0행, 예측=1열) 칸의 값이 크다면, 클래스 0을 클래스 1로 자주 착각한다는 뜻입니다.

### 💻 예제 코드

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=sorted(y.unique()), yticklabels=sorted(y.unique()))
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('RandomForest Confusion Matrix (3 classes)')
plt.show()

### ✍️ TODO: 실전 문제

**문제 1.** `LogisticRegression` 예측 결과의 3x3 confusion matrix를 heatmap으로 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
logi_pred = preds['LogisticRegression']
cm2 = confusion_matrix(y_test, logi_pred)  # 클래스가 3개이므로 3x3 행렬이 반환됨
sns.heatmap(cm2, annot=True, fmt='d', cmap='Greens')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.show()
```

</details>

## 6. 앙상블 개념 복습 + 다중분류 실전 비교 (배깅 vs 부스팅)

### 📖 이론 설명
배깅(RandomForest)과 부스팅(XGBoost, LGBM)의 개념 자체는 이진분류(06 노트북 3절)와 동일합니다. 다중분류에서 실제로 확인해볼 점은, 클래스가 3개로 늘어나도 두 방식이 여전히 잘 작동하는지를 **macro f1 기준으로 나란히 비교**해보는 것입니다.

### 💻 예제 코드

In [ ]:
from sklearn.metrics import f1_score
for name in ['RandomForest', 'XGBoost']:
    f1 = f1_score(y_test, preds[name], average='macro')
    print(f'{name} (macro f1): {f1:.4f}')

### ✍️ TODO: 실전 문제

**문제 1.** `GradientBoostingClassifier(n_estimators=100, random_state=42)`를 추가로 학습시키고, `RandomForest`/`XGBoost`와 macro f1을 함께 비교하세요.

In [ ]:
# 여기에 코드를 작성하세요

<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.ensemble import GradientBoostingClassifier
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb_model.fit(X_train_scaled, y_train)
preds['GradientBoosting'] = gb_model.predict(X_test_scaled)
for name in ['RandomForest', 'XGBoost', 'GradientBoosting']:
    print(name, f1_score(y_test, preds[name], average='macro'))
```

</details>

## 7. Feature Importance 해석

### 📖 이론 설명
트리 계열 모델의 `feature_importances_`로 와인의 어떤 화학 성분이 등급(target) 예측에 가장 크게 기여했는지 확인할 수 있습니다.

### 💻 예제 코드

In [ ]:
rf_model = models['RandomForest']
rf_imp = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(rf_imp)
plt.barh(rf_imp.index, rf_imp.values)
plt.title('Feature Importance (RandomForest)')
plt.show()

### ✍️ TODO: 실전 문제

**문제 1.** `XGBoost` 모델의 feature_importances_ 상위 3개 변수 이름을 `top3_features`에 저장하세요.

In [ ]:
# 여기에 코드를 작성하세요

<details>
<summary>✅ 정답 보기</summary>

```python
xgb_model = models['XGBoost']
xgb_imp = pd.Series(xgb_model.feature_importances_, index=X.columns).sort_values(ascending=False)
top3_features = xgb_imp.head(3).index.tolist()
print(top3_features)
```

</details>

## 8. 여러 모델 앙상블 (VotingClassifier)

### 📖 이론 설명
이진분류와 마찬가지로, 서로 다른 모델의 예측을 `VotingClassifier`로 종합하면 개별 모델의 약점이 상쇄되어 더 안정적인 예측을 얻을 수 있습니다. 다중분류에서도 `voting='soft'`는 각 모델이 예측한 클래스별 확률(`predict_proba`)의 평균으로 최종 클래스를 정합니다.

### 💻 예제 코드

In [ ]:
from sklearn.ensemble import VotingClassifier

voting_models = [('logi', LogisticRegression(max_iter=1000)),
                 ('rf', RandomForestClassifier(n_estimators=100, random_state=42))]
voting = VotingClassifier(voting_models, voting='soft')
voting.fit(X_train_scaled, y_train)
voting_pred = voting.predict(X_test_scaled)
print('Voting macro f1:', f1_score(y_test, voting_pred, average='macro'))

### ✍️ TODO: 실전 문제

**문제 1.** `voting_models`에 `XGBoost`(`models['XGBoost']`)까지 포함한 3개 모델짜리 `voting2`(`voting='soft'`)를 만들어 macro f1을 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요

<details>
<summary>✅ 정답 보기</summary>

```python
voting_models2 = [('logi', LogisticRegression(max_iter=1000)),
                  ('rf', RandomForestClassifier(n_estimators=100, random_state=42)),
                  ('xgb', XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss'))]
voting2 = VotingClassifier(voting_models2, voting='soft')
voting2.fit(X_train_scaled, y_train)
voting2_pred = voting2.predict(X_test_scaled)
print('Voting(3모델) macro f1:', f1_score(y_test, voting2_pred, average='macro'))
```

</details>

## 9. 여러 모델 성능 비교 시각화

### 📖 이론 설명
모델을 하나씩 따로 보는 것보다, 여러 모델의 macro f1을 한 번에 막대그래프로 나란히 비교하면 어떤 모델이 이 데이터에 더 잘 맞는지 빠르게 파악할 수 있습니다.

### 💻 예제 코드

In [ ]:
score_table = pd.DataFrame({
    name: {'accuracy': accuracy_score(y_test, pred),
           'f1_macro': f1_score(y_test, pred, average='macro')}
    for name, pred in preds.items()
}).T.sort_values('f1_macro', ascending=False)
print(score_table)

score_table['f1_macro'].plot(kind='barh')
plt.title('Model Comparison - Macro F1 Score')
plt.show()

### ✍️ TODO: 실전 문제

**문제 1.** `score_table`에 `Voting`(문제 8에서 만든 `voting_pred`) 행을 추가한 뒤, accuracy 기준으로 다시 정렬해서 출력하세요.

In [ ]:
# 여기에 코드를 작성하세요

<details>
<summary>✅ 정답 보기</summary>

```python
score_table.loc['Voting'] = {
    'accuracy': accuracy_score(y_test, voting_pred),
    'f1_macro': f1_score(y_test, voting_pred, average='macro'),
}
print(score_table.sort_values('accuracy', ascending=False))
```

</details>

## 10. 소프트맥스 직관 & 다중클래스 레이블 인코딩

### 📖 이론 설명
이진분류의 `LogisticRegression`은 시그모이드로 '하나의 확률'만 계산하지만, 다중분류에서는 내부적으로 **소프트맥스(softmax)**를 사용해 **클래스 개수만큼의 확률**을 한 번에 계산하고, 그중 가장 높은 확률의 클래스를 최종 예측으로 선택합니다. 따라서 `predict_proba()`의 각 행(row)을 모두 더하면 항상 1이 됩니다.

또한 `target`이 지금처럼 이미 0,1,2 정수라면 상관없지만, 실제 데이터는 `'setosa'`, `'versicolor'`처럼 **문자열 레이블**로 되어 있는 경우가 많습니다. sklearn 모델은 문자열 레이블을 대부분 그대로 학습할 수 있지만, 나중에 다시 원래 이름으로 되돌리거나 다른 라이브러리와 맞추려면 `LabelEncoder`로 정수 ↔ 문자열을 변환하는 방법을 알아두는 것이 좋습니다.

### 💻 예제 코드

In [ ]:
logi_model = models['LogisticRegression']
proba = logi_model.predict_proba(X_test_scaled)
print(proba[:3])  # 클래스 3개에 대한 확률
print(proba[:3].sum(axis=1))  # 각 행의 합은 항상 1(소프트맥스의 성질)

### ✍️ TODO: 실전 문제

**문제 1.** `wine['target']`(0,1,2)을 `{'class_0','class_1','class_2'}` 문자열로 바꾼 `label_str`을 만들고, `LabelEncoder`로 다시 정수로 인코딩한 `label_encoded`가 원래 `target`과 같은지 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요

<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.preprocessing import LabelEncoder

label_str = wine['target'].map({0: 'class_0', 1: 'class_1', 2: 'class_2'})
le = LabelEncoder()
label_encoded = le.fit_transform(label_str)
print(le.classes_)  # 인코더가 학습한 클래스 목록(알파벳 순)
print((label_encoded == wine['target']).all())  # True면 원래 정수 라벨과 순서까지 동일
```

</details>

## 11. 종합 실전 문제

**다중분류 전체 흐름을 이어서 풀어보는 미니 모의고사입니다.**

**문제 1.** `RandomForestClassifier(n_estimators=200, random_state=42)`를 학습시키고 `classification_report`를 출력하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.metrics import classification_report
rf2 = RandomForestClassifier(n_estimators=200, random_state=42)
rf2.fit(X_train_scaled, y_train)
print(classification_report(y_test, rf2.predict(X_test_scaled)))  # classification_report는 옵션 없이도 클래스별 precision/recall/f1을 모두 보여줘서 다중분류에서 가장 먼저 찍어보기 좋음
```

</details>

**문제 2.** `RandomForest` 모델의 feature_importances_ 중 가장 중요한 변수 이름을 `top_feature`에 저장하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
imp = pd.Series(models['RandomForest'].feature_importances_, index=X.columns)  # feature_importances_ 배열에 컬럼명을 붙여 어떤 값이 어떤 변수인지 알 수 있게 함
top_feature = imp.idxmax()
print(top_feature)
```

</details>